# LLM Eval System — one-shot real run on Colab free T4

End-to-end: installs the project, boots vLLM on a small open-weight model,
runs Parts B / C / E to produce **real** numbers, zips everything up, and
triggers a browser download so you can scp the artefacts into your repo
and commit them.

**Before you run this:**

1. `Runtime` → `Change runtime type` → **T4 GPU** (any GPU works;
   T4 is the free tier).
2. Edit the `REPO_URL` + `MODEL` cells below if you want a different repo /
   model. Defaults:
   - Repo: your fork on GitHub (set `REPO_URL` to your git URL).
   - Model: `Qwen/Qwen2.5-1.5B-Instruct` — strong, small, **not gated**
     so no HF token needed. Alternatives:
     - `microsoft/Phi-3-mini-4k-instruct` (3.8 B, also ungated, higher accuracy)
     - `meta-llama/Llama-3.2-1B-Instruct` (gated — needs `HF_TOKEN`)

3. `Runtime` → `Run all`. Total wall time ~45–60 min. Leave the tab
   open (Colab idles you out if the tab is backgrounded too long).

The last cell downloads `llmeval_results.tgz` to your laptop. Commit
its contents to your repo and you're done.

## 1. Config — edit these

In [ ]:
REPO_URL = "https://github.com/<YOUR-GITHUB-USERNAME>/LLMEvalSystem.git"
BRANCH = "main"

# Pick a model. Un-gated defaults avoid HF_TOKEN friction.
# For Llama-3.2, set HF_TOKEN = "hf_xxxx" below and accept the license first
# at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

HF_TOKEN = ""  # only needed for gated models

# How many examples per task. 200 gives tight CIs in ~10-20 min;
# 500 is ideal for a final report; use 50 for fast iteration.
N_EVAL = 200

# vLLM serving args — safe defaults for a T4.
MAX_MODEL_LEN = 2048
DTYPE = "float16"            # T4 is Turing (SM 7.5) -- no native bf16. Use 'bfloat16' only on A100/H100.
GPU_MEM_UTIL = 0.85

## 2. Confirm we actually have a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 3. Install uv + clone the project

In [ ]:
import os, subprocess

# uv (fast Python dep manager). Persist in PATH for later cells.
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
!uv --version

# Clone if not already present.
if not os.path.isdir("LLMEvalSystem"):
    !git clone --branch $BRANCH $REPO_URL LLMEvalSystem
os.chdir("LLMEvalSystem")
!git log --oneline -1

# Full install -- all extras in one shot so torch/vllm/lm-eval land together.
# This takes ~5-10 min cold; cached across re-runs inside the same session.
!uv sync --extra serve --extra eval --extra perf --extra improve

## 4. (Optional) HuggingFace login for gated models

In [ ]:
if HF_TOKEN:
    !uv run huggingface-cli login --token $HF_TOKEN --add-to-git-credential
else:
    print("HF_TOKEN empty -- gated models (e.g. Llama-3.2) will fail to download.")
    print("If you're using a gated model, paste a token in the config cell and rerun.")

## 5. Configure Settings via env vars

In [ ]:
os.environ.update({
    "LLMEVAL_MOCK_BACKEND": "false",
    "LLMEVAL_MODEL_NAME": MODEL,
    "LLMEVAL_VLLM_HOST": "127.0.0.1",
    "LLMEVAL_VLLM_PORT": "8000",
    "LLMEVAL_VLLM_API_KEY": "EMPTY",
    "LLMEVAL_VLLM_DTYPE": DTYPE,
    "LLMEVAL_VLLM_MAX_MODEL_LEN": str(MAX_MODEL_LEN),
    "LLMEVAL_VLLM_GPU_MEMORY_UTILIZATION": str(GPU_MEM_UTIL),
    "LLMEVAL_VLLM_TIMEOUT_S": "300",
    "LLMEVAL_SEED": "42",
    "LLMEVAL_LOG_FORMAT": "json",
    # Colab-friendliness: force the spawn multiproc method so the worker
    # subprocesses don't inherit a live CUDA context from the kernel
    # (common cause of 'Engine core initialization failed' on free T4).
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
})

# Sanity: mock-backend test suite still works with the new settings.
!uv run pytest tests/common -q

## 6. Boot vLLM in the background and wait for it to be ready

The first run downloads the model weights from HuggingFace (one-time
~2-8 GB depending on the model). Subsequent runs in the same Colab session
are cached.

We don't use `make serve` here because we need the Python process handle
to kill it cleanly after the eval + perf runs.

In [ ]:
import subprocess, time, urllib.request, urllib.error

serve_log = open("vllm_serve.log", "w")
serve_proc = subprocess.Popen(
    ["uv", "run", "python", "-m", "serve.serve"],
    stdout=serve_log, stderr=subprocess.STDOUT,
)
print(f"vLLM pid={serve_proc.pid}, log=vllm_serve.log, tailing readiness ...")

# Poll /v1/models until it responds, then we're ready.
url = "http://127.0.0.1:8000/v1/models"
deadline = time.monotonic() + 900  # 15 min cap for the first weight download
ready = False
while time.monotonic() < deadline:
    if serve_proc.poll() is not None:
        print("!! vLLM exited early. Last 40 log lines:")
        !tail -n 40 vllm_serve.log
        raise RuntimeError("vLLM failed to start.")
    try:
        req = urllib.request.Request(url, headers={"Authorization": "Bearer EMPTY"})
        resp = urllib.request.urlopen(req, timeout=3)
        if resp.status == 200:
            ready = True
            break
    except (urllib.error.URLError, ConnectionRefusedError, TimeoutError):
        pass
    time.sleep(5)

if not ready:
    raise RuntimeError("vLLM didn't come up in 15 min. See vllm_serve.log.")
print("vLLM is serving. Moving on.")

## 7. Sanity: one quick completion to prove end-to-end

In [ ]:
!uv run python -m serve.client "What is the capital of France?" --max-tokens 16

## 8. Part B — lm-eval on MMLU-STEM + HellaSwag + custom_qa

In [ ]:
!uv run python -m eval_runner.run_eval \
    --task mmlu_stem,hellaswag,custom_qa \
    --limit $N_EVAL \
    --max-concurrency 16 \
    --method real_gpu_$MODEL \
    --notes "one-shot Colab run on T4"

## 9. Part C — load test + GPU monitor

In [ ]:
import subprocess

# Start GPU sampler in the background.
gpu_proc = subprocess.Popen(
    ["uv", "run", "python", "-m", "perf.gpu_monitor",
     "--output", "results/perf/gpu.csv",
     "--interval", "1", "--duration", "180"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Load test.
!uv run python -m perf.load_test \
    --num-requests 200 --concurrency 16 --mode stream \
    --output results/perf/metrics.csv

# Summarise.
!uv run python -m perf.metrics --input results/perf/metrics.csv --summary results/perf/summary.csv

gpu_proc.terminate()
try:
    gpu_proc.wait(timeout=5)
except subprocess.TimeoutExpired:
    gpu_proc.kill()

!cat results/perf/summary.csv

## 10. Part E — full HellaSwag ablation ladder

One row per variant, paired-bootstrap CI vs baseline, all rows landing in
`results/result-log.csv` and `docs/improvement-log.md`.

In [ ]:
os.environ["N_EVAL"] = str(N_EVAL)
os.environ["N_POOL"] = "500"
os.environ["MAX_CONCURRENCY"] = "16"

!bash improve/eval.sh

## 11. Determinism check (Part D)

In [ ]:
!uv run python -m guardrails.validate determinism \
    "The capital of France is" --n-runs 5 --max-tokens 16 --seed 42 \
    --report results/determinism_report.json
!cat results/determinism_report.json

## 12. Inspect the final tables

In [ ]:
print("=== results/summary.md ===")
!cat results/summary.md
print()
print("=== results/result-log.md ===")
!cat results/result-log.md
print()
print("=== docs/improvement-log.md (tail) ===")
!tail -n 80 docs/improvement-log.md

## 13. Shut vLLM down and package results

In [ ]:
import signal
serve_proc.send_signal(signal.SIGTERM)
try:
    serve_proc.wait(timeout=30)
except subprocess.TimeoutExpired:
    serve_proc.kill()
print("vLLM stopped.")

!tar czf /tmp/llmeval_results.tgz \
    results/ \
    docs/improvement-log.md \
    improve/report.md \
    vllm_serve.log

!ls -lh /tmp/llmeval_results.tgz

## 14. Download the tarball to your laptop

Unpack it into your repo checkout, commit, and submit.

In [ ]:
from google.colab import files
files.download("/tmp/llmeval_results.tgz")

### Back on your laptop

```bash
cd path/to/LLMEvalSystem
tar xzf ~/Downloads/llmeval_results.tgz
git add results/ docs/improvement-log.md improve/report.md
git status
git commit -m "chore: real numbers from Colab T4 run on <model>"
```

Your `improve/report.md` results table now has real numbers; submit the
repo per the problem-statement submission instructions.